# Lecture 4: the household that lives forever

**ECU44291 Quantitative Macroeconomics, Week 4.**

Drop the age from last week's Bellman equation and there is no terminal condition to start from. The equation becomes a fixed-point problem in a *function*:
$$V(k) = \max_{0 < k' < k^{\alpha}} \; \ln(k^{\alpha} - k') + \beta V(k').$$
This is the neoclassical growth model with log utility and full depreciation. It has a closed form, $V(k) = A + B\ln k$ and $g(k) = \alpha\beta k^{\alpha}$, which is our check.

Value function iteration is Week 1's fixed-point loop applied to $V$: guess, apply the operator, repeat until nothing changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

alpha, beta = 0.3, 0.9
k = np.linspace(0.02, 0.5, 200)

def closed_form_value(k):
    B = alpha / (1 - alpha * beta)
    A = (1 / (1 - beta)) * ((alpha * beta / (1 - alpha * beta)) * np.log(alpha * beta) + np.log(1 - alpha * beta))
    return A + B * np.log(k)

def closed_form_policy(k):
    return alpha * beta * k ** alpha

## Warm-up: two sweeps by hand

The slide's cake: sizes $w \in \{0, 1, 2\}$, $u(c) = \sqrt{c}$, $\beta = 0.9$, $V_0 = 0$. Do sweeps 1 and 2 on paper first, then run the cell to check. Every infeasible choice (keeping more than the cake) gets $-\infty$.

In [ ]:
w = np.array([0, 1, 2])
c_cake = w[:, None] - w[None, :]                  # rows: w today; columns: w' kept
u_cake = np.full(c_cake.shape, -np.inf)
u_cake[c_cake >= 0] = np.sqrt(c_cake[c_cake >= 0])
V_cake = np.zeros(3)
for sweep in (1, 2, 3):
    value = u_cake + 0.9 * V_cake[None, :]
    V_cake, keep = value.max(axis=1), value.argmax(axis=1)
    print(f"sweep {sweep}: V = {np.round(V_cake, 3)}, keep w' = {keep}")

## Task 1: the Bellman operator

Compute the utility matrix `U[i, j] = ln(k_i^alpha - k_j)` once (with $-\infty$ where consumption is not positive). Then write `bellman(V)` returning `(TV, policy)` where `TV[i] = max_j U[i, j] + beta * V[j]`. One line each with NumPy broadcasting. Why compute `U` once rather than inside the loop?

In [ ]:
# SOLUTION
c = k[:, None] ** alpha - k[None, :]
U = np.full(c.shape, -np.inf)
U[c > 0] = np.log(c[c > 0])

def bellman(V):
    value = U + beta * V[None, :]
    return value.max(axis=1), value.argmax(axis=1)

TV, pol = bellman(np.zeros(len(k)))
print(TV[:3], pol[:3])

## Task 2: value function iteration

Write the loop: start from $V_0 = 0$, apply `bellman`, measure the sup-norm distance, stop below $10^{-6}$. Keep every iterate in a list so we can plot them. Report the number of sweeps. Then plot the iterates and the closed form on one figure.

In [ ]:
# SOLUTION
def vfi(tol=1e-6, max_iter=10000):
    V = np.zeros(len(k))
    history, dists = [V.copy()], []
    for it in range(max_iter):
        V_new, policy = bellman(V)
        dist = np.max(np.abs(V_new - V))
        dists.append(dist)
        V = V_new
        history.append(V.copy())
        if dist < tol:
            return V, policy, np.array(dists), history
    raise RuntimeError("no convergence")

V, policy, dists, history = vfi()
print(f"{len(dists)} sweeps; max |V - closed form| = {np.max(np.abs(V - closed_form_value(k))):.4f}")
for Vn in history[::5]:
    plt.plot(k, Vn, color="tab:blue", lw=0.7, alpha=0.5)
plt.plot(k, closed_form_value(k), "--", color="tab:orange", lw=2, label="closed form")
plt.legend(); plt.xlabel("k"); plt.ylabel("V"); plt.show()

**Workflow note.** *If it converged in one sweep, it did not converge.* A first sweep from $V_0 = 0$ must change something. `git diff` shows the line that stopped it changing; `git checkout -- src/` takes it back.

## Task 3: the rate

Plot the distances on a log scale. What is the ratio of consecutive distances after the first few sweeps? Compare it to $\beta$. The number of sweeps to tolerance is about $\ln(\text{tol}/d_0)/\ln\beta$, with $d_0$ the first distance: check it at $\beta = 0.9$, then predict how many sweeps $\beta = 0.99$ needs, and check that too.

In [ ]:
# SOLUTION
plt.semilogy(dists); plt.xlabel("sweep"); plt.ylabel("sup-norm change"); plt.show()
print("late ratios:", np.round(dists[-5:] / dists[-6:-1], 4), " beta =", beta)
print(f"beta = 0.9: predicted about {int(np.ceil(np.log(1e-6 / dists[0]) / np.log(0.9)))} sweeps, actual {len(dists)}")
predicted = int(np.ceil(np.log(1e-6 / dists[0]) / np.log(0.99)))
beta_saved, beta = beta, 0.99
V99, _, d99, _ = vfi()
beta = beta_saved
print(f"beta = 0.99: predicted about {predicted} sweeps, actual {len(d99)}")

## Task 4: the policy

Plot the grid policy $k' = k[\text{policy}]$ against the closed form $\alpha\beta k^{\alpha}$ and the 45-degree line. Where is the steady state? Why is the grid policy a staircase?

In [ ]:
# SOLUTION
kstar = (alpha * beta) ** (1 / (1 - alpha))
plt.step(k, k[policy], where="mid", label="grid policy")
plt.plot(k, closed_form_policy(k), "--", label="closed form")
plt.plot(k, k, "k:", lw=0.8)
plt.plot(kstar, kstar, "ko"); plt.legend(); plt.xlabel("k"); plt.ylabel("k'"); plt.show()
print(f"k* = {kstar:.4f}")

## Task 5: Howard's improvement

Instead of improving $V$ a little each sweep, take a policy, compute its value *exactly* by solving the linear system $(I - \beta P_\sigma) v = r_\sigma$, then take the greedy policy for that value, and repeat until the policy stops changing. Write `policy_evaluation(policy)` and the loop. How many steps?

In [ ]:
# SOLUTION
def policy_evaluation(policy):
    n = len(policy)
    r = U[np.arange(n), policy]
    P = np.zeros((n, n)); P[np.arange(n), policy] = 1.0
    return np.linalg.solve(np.eye(n) - beta * P, r)

def howard(max_iter=100):
    policy = np.zeros(len(k), dtype=int)
    for it in range(max_iter):
        v = policy_evaluation(policy)
        _, new_policy = bellman(v)
        if np.array_equal(new_policy, policy):
            return v, policy, it + 1
        policy = new_policy
    raise RuntimeError("no convergence")

vH, polH, nH = howard()
print(f"Howard: {nH} steps; same policy as VFI: {np.array_equal(polH, policy)}; max |vH - V| = {np.max(np.abs(vH - V)):.2e}")

## Task 6: time it

Time `vfi()` and `howard()`. Then write the naive version with three explicit Python loops and utility recomputed inside (the way the MSc notebook did it), run it with a 50-point grid, and time that. Measure before you optimise: which change bought the most?

In [ ]:
# SOLUTION
t0 = time.perf_counter(); vfi(); t_vfi = time.perf_counter() - t0
t0 = time.perf_counter(); howard(); t_how = time.perf_counter() - t0
print(f"vectorised VFI {t_vfi:.3f} s, Howard {t_how:.3f} s on {len(k)} grid points")

def vfi_naive(kk, tol=1e-6):
    V = np.zeros(len(kk))
    for it in range(10000):
        V_new = np.empty(len(kk))
        for i in range(len(kk)):
            best = -np.inf
            for j in range(len(kk)):
                cc = kk[i] ** alpha - kk[j]
                if cc > 0:
                    best = max(best, np.log(cc) + beta * V[j])
            V_new[i] = best
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, it + 1
        V = V_new

k50 = np.linspace(0.02, 0.5, 50)
t0 = time.perf_counter(); vfi_naive(k50); t_naive = time.perf_counter() - t0
print(f"naive loops on 50 points: {t_naive:.2f} s")

## Before next week

- Put `bellman`, `vfi`, `policy_evaluation`, `howard` in `src/vfi.py` with a test against the closed form. Commit.
- Keep working on M1 Part A. The solo window is open; Copilot is off.
- Read method card 6 (discrete VFI with Howard).
- Workflow note: if VFI converges in one sweep with distance exactly zero, it did not converge. `git diff` shows you the one line.